# Project 1 · Part 3: Project 1 · Part 3: Credit risk — a default model and an approval policy · Practice

**Open Fraud Labs Academy**

1. Copy your **practice key** from the Practice tab of this lesson on the Academy and paste it into the first code cell.
2. Run the first cell, then complete each exercise and run its check.
3. Every check that passes is saved to your lesson page automatically. Solve all three to unlock the quiz.

Tip: choose **File > Save a copy in Drive** to keep your work.

In [ ]:
# Step 1: paste your practice key between the quotes, then run this cell (Shift + Enter).
# Find your key on the lesson page, in the Practice tab.
OFL_KEY = ""

import pandas as pd, requests
DATA_URL = "https://raw.githubusercontent.com/Odugbile1993/openfraudlab-tiktok/main/data/loans.csv"
COURSE, LESSON = "data-science", 33

def ofl_check(exercise, check_code):
    """Runs the check for one exercise and, if it passes, records it on your Academy lesson page."""
    try:
        exec(check_code, globals())
    except AssertionError as e:
        print("Not quite yet:", e or "compare your result with the task and try again.")
        return
    except NameError as e:
        print("Something the task asks for is missing. Run your code cell first.", e)
        return
    print("Correct! Well done.")
    if not OFL_KEY.strip():
        print("Add your practice key in the first cell and run it, then run this check again to record your progress.")
        return
    try:
        r = requests.post("https://virxrqwxvgsbcnmhdwlp.supabase.co/rest/v1/rpc/practice_report",
                          headers={"apikey": "sb_publishable_8IenXF_4F5ew8hLai8RfVA_dNHq4I72", "Content-Type": "application/json"},
                          json={"p_key": OFL_KEY.strip(), "p_course": COURSE, "p_lesson": LESSON, "p_exercise": exercise}, timeout=20)
        d = r.json()
        if r.ok:
            print(f"Saved to your lesson page: {d['solved']} of {d['total']} exercises solved.")
        else:
            print("Could not save your progress:", d.get("message", r.text))
    except Exception as e:
        print("Could not reach the Academy to save your progress:", e)

print("Ready. Work through the exercises below." if OFL_KEY.strip() else "Paste your practice key above so your progress is saved.")

## Exercise 1: ROC AUC of the balanced model

Fit LogisticRegression(max_iter=1000, class_weight="balanced") on the training data. Score the test loans and store the ROC AUC in auc.

<details><summary>Hint</summary>

Fit with .fit(X_train, y_train). roc_auc_score takes the true labels and the scores: roc_auc_score(y_test, scores).

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

credit = pd.read_csv(DATA_URL.replace("loans.csv", "projects/credit_applications.csv"))
credit["employment_type"] = credit["employment_type"].fillna("Unknown")
too_long = credit["months_at_employer"] > (credit["age"] - 16) * 12
credit.loc[too_long, "months_at_employer"] = None
for col in ["months_at_employer", "digital_payments_per_month"]:
    credit[col] = credit[col].fillna(credit[col].median())
credit["dti"] = (credit["monthly_repayment_usd"] + credit["existing_debt_usd"] / 12) / credit["monthly_income_usd"]

features = ["dti", "previous_late_payments", "previous_loans", "has_salary_account", "has_guarantor",
            "months_at_employer", "age", "tenure_months", "digital_payments_per_month",
            "employment_type", "loan_purpose"]
X = pd.get_dummies(credit[features], drop_first=True, dtype=int)
y = credit["defaulted"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
from sklearn.metrics import roc_auc_score

balanced = ...
scores = balanced.predict_proba(X_test)[:, 1]
auc = ...
print(auc)

In [ ]:
# Check exercise 1 (run after your code above)
ofl_check(1, "_ns = {\"DATA_URL\": DATA_URL}\nexec('import pandas as pd\\nfrom sklearn.model_selection import train_test_split\\nfrom sklearn.linear_model import LogisticRegression\\n\\ncredit = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"projects/credit_applications.csv\"))\\ncredit[\"employment_type\"] = credit[\"employment_type\"].fillna(\"Unknown\")\\ntoo_long = credit[\"months_at_employer\"] > (credit[\"age\"] - 16) * 12\\ncredit.loc[too_long, \"months_at_employer\"] = None\\nfor col in [\"months_at_employer\", \"digital_payments_per_month\"]:\\n    credit[col] = credit[col].fillna(credit[col].median())\\ncredit[\"dti\"] = (credit[\"monthly_repayment_usd\"] + credit[\"existing_debt_usd\"] / 12) / credit[\"monthly_income_usd\"]\\n\\nfeatures = [\"dti\", \"previous_late_payments\", \"previous_loans\", \"has_salary_account\", \"has_guarantor\",\\n            \"months_at_employer\", \"age\", \"tenure_months\", \"digital_payments_per_month\",\\n            \"employment_type\", \"loan_purpose\"]\\nX = pd.get_dummies(credit[features], drop_first=True, dtype=int)\\ny = credit[\"defaulted\"]\\nX_train, X_test, y_train, y_test = train_test_split(\\n    X, y, test_size=0.25, random_state=42, stratify=y)\\nbalanced = LogisticRegression(max_iter=1000, class_weight=\"balanced\").fit(X_train, y_train)\\n', _ns)\nfrom sklearn.metrics import roc_auc_score as _auc\n_want = _auc(_ns[\"y_test\"], _ns[\"balanced\"].predict_proba(_ns[\"X_test\"])[:, 1])\nassert auc is not Ellipsis and abs(float(auc) - _want) < 1e-6, \"auc should be roc_auc_score(y_test, scores) for LogisticRegression(max_iter=1000, class_weight=\\\"balanced\\\")\"\n")

## Exercise 2: A stricter cut-off

Flag test loans whose risk score is 0.7 or higher. Store the recall in rec and the precision in prec. Compare them with the 0.5 cut-off from the lesson (recall 0.477, precision 0.133).

<details><summary>Hint</summary>

flagged = scores >= 0.7 gives True/False for each loan; recall_score(y_test, flagged) and precision_score(y_test, flagged).

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

credit = pd.read_csv(DATA_URL.replace("loans.csv", "projects/credit_applications.csv"))
credit["employment_type"] = credit["employment_type"].fillna("Unknown")
too_long = credit["months_at_employer"] > (credit["age"] - 16) * 12
credit.loc[too_long, "months_at_employer"] = None
for col in ["months_at_employer", "digital_payments_per_month"]:
    credit[col] = credit[col].fillna(credit[col].median())
credit["dti"] = (credit["monthly_repayment_usd"] + credit["existing_debt_usd"] / 12) / credit["monthly_income_usd"]

features = ["dti", "previous_late_payments", "previous_loans", "has_salary_account", "has_guarantor",
            "months_at_employer", "age", "tenure_months", "digital_payments_per_month",
            "employment_type", "loan_purpose"]
X = pd.get_dummies(credit[features], drop_first=True, dtype=int)
y = credit["defaulted"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
balanced = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train, y_train)
from sklearn.metrics import precision_score, recall_score

scores = balanced.predict_proba(X_test)[:, 1]
flagged = ...
rec = ...
prec = ...
print(rec, prec)

In [ ]:
# Check exercise 2 (run after your code above)
ofl_check(2, "_ns = {\"DATA_URL\": DATA_URL}\nexec('import pandas as pd\\nfrom sklearn.model_selection import train_test_split\\nfrom sklearn.linear_model import LogisticRegression\\n\\ncredit = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"projects/credit_applications.csv\"))\\ncredit[\"employment_type\"] = credit[\"employment_type\"].fillna(\"Unknown\")\\ntoo_long = credit[\"months_at_employer\"] > (credit[\"age\"] - 16) * 12\\ncredit.loc[too_long, \"months_at_employer\"] = None\\nfor col in [\"months_at_employer\", \"digital_payments_per_month\"]:\\n    credit[col] = credit[col].fillna(credit[col].median())\\ncredit[\"dti\"] = (credit[\"monthly_repayment_usd\"] + credit[\"existing_debt_usd\"] / 12) / credit[\"monthly_income_usd\"]\\n\\nfeatures = [\"dti\", \"previous_late_payments\", \"previous_loans\", \"has_salary_account\", \"has_guarantor\",\\n            \"months_at_employer\", \"age\", \"tenure_months\", \"digital_payments_per_month\",\\n            \"employment_type\", \"loan_purpose\"]\\nX = pd.get_dummies(credit[features], drop_first=True, dtype=int)\\ny = credit[\"defaulted\"]\\nX_train, X_test, y_train, y_test = train_test_split(\\n    X, y, test_size=0.25, random_state=42, stratify=y)\\nbalanced = LogisticRegression(max_iter=1000, class_weight=\"balanced\").fit(X_train, y_train)\\n', _ns)\nfrom sklearn.metrics import recall_score as _rs, precision_score as _ps\n_f = _ns[\"balanced\"].predict_proba(_ns[\"X_test\"])[:, 1] >= 0.7\nassert rec is not Ellipsis and abs(float(rec) - _rs(_ns[\"y_test\"], _f)) < 1e-9, \"rec should be recall_score(y_test, flagged) with flagged = scores >= 0.7\"\nassert prec is not Ellipsis and abs(float(prec) - _ps(_ns[\"y_test\"], _f)) < 1e-9, \"prec should be precision_score(y_test, flagged)\"\n")

## Exercise 3: Test an approval policy

Approve test loans with a risk score below 0.6. Store the share approved in approval_rate, the default rate among approved loans in default_rate, and the estimated profit (using the example profit function) in profit_usd.

<details><summary>Hint</summary>

approve = scores < 0.6. approve.mean() is the approval rate; y_test[approve].mean() is the default rate among approved loans.

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

credit = pd.read_csv(DATA_URL.replace("loans.csv", "projects/credit_applications.csv"))
credit["employment_type"] = credit["employment_type"].fillna("Unknown")
too_long = credit["months_at_employer"] > (credit["age"] - 16) * 12
credit.loc[too_long, "months_at_employer"] = None
for col in ["months_at_employer", "digital_payments_per_month"]:
    credit[col] = credit[col].fillna(credit[col].median())
credit["dti"] = (credit["monthly_repayment_usd"] + credit["existing_debt_usd"] / 12) / credit["monthly_income_usd"]

features = ["dti", "previous_late_payments", "previous_loans", "has_salary_account", "has_guarantor",
            "months_at_employer", "age", "tenure_months", "digital_payments_per_month",
            "employment_type", "loan_purpose"]
X = pd.get_dummies(credit[features], drop_first=True, dtype=int)
y = credit["defaulted"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
balanced = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train, y_train)

MARGIN = 0.20   # example assumption, not a real figure
def profit(approve, defaulted, amount):
    gain = MARGIN * amount[approve & (defaulted == 0)].sum()
    loss = amount[approve & (defaulted == 1)].sum()
    return gain - loss

amount_test = credit.loc[X_test.index, "loan_amount_usd"]
scores = balanced.predict_proba(X_test)[:, 1]
approve = ...
approval_rate = ...
default_rate = ...
profit_usd = ...
print(approval_rate, default_rate, profit_usd)

In [ ]:
# Check exercise 3 (run after your code above)
ofl_check(3, "_ns = {\"DATA_URL\": DATA_URL}\nexec('import pandas as pd\\nfrom sklearn.model_selection import train_test_split\\nfrom sklearn.linear_model import LogisticRegression\\n\\ncredit = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"projects/credit_applications.csv\"))\\ncredit[\"employment_type\"] = credit[\"employment_type\"].fillna(\"Unknown\")\\ntoo_long = credit[\"months_at_employer\"] > (credit[\"age\"] - 16) * 12\\ncredit.loc[too_long, \"months_at_employer\"] = None\\nfor col in [\"months_at_employer\", \"digital_payments_per_month\"]:\\n    credit[col] = credit[col].fillna(credit[col].median())\\ncredit[\"dti\"] = (credit[\"monthly_repayment_usd\"] + credit[\"existing_debt_usd\"] / 12) / credit[\"monthly_income_usd\"]\\n\\nfeatures = [\"dti\", \"previous_late_payments\", \"previous_loans\", \"has_salary_account\", \"has_guarantor\",\\n            \"months_at_employer\", \"age\", \"tenure_months\", \"digital_payments_per_month\",\\n            \"employment_type\", \"loan_purpose\"]\\nX = pd.get_dummies(credit[features], drop_first=True, dtype=int)\\ny = credit[\"defaulted\"]\\nX_train, X_test, y_train, y_test = train_test_split(\\n    X, y, test_size=0.25, random_state=42, stratify=y)\\nbalanced = LogisticRegression(max_iter=1000, class_weight=\"balanced\").fit(X_train, y_train)\\n', _ns)\n_yt = _ns[\"y_test\"]\n_a = _ns[\"balanced\"].predict_proba(_ns[\"X_test\"])[:, 1] < 0.6\n_amt = _ns[\"credit\"].loc[_ns[\"X_test\"].index, \"loan_amount_usd\"]\n_p = 0.20 * _amt[_a & (_yt == 0)].sum() - _amt[_a & (_yt == 1)].sum()\nassert approval_rate is not Ellipsis and abs(float(approval_rate) - _a.mean()) < 1e-9, \"approval_rate should be the share approved: approve.mean() with approve = scores < 0.6\"\nassert default_rate is not Ellipsis and abs(float(default_rate) - _yt[_a].mean()) < 1e-9, \"default_rate should be the default rate among approved loans: y_test[approve].mean()\"\nassert profit_usd is not Ellipsis and abs(float(profit_usd) - _p) < 1e-3, \"profit_usd should be profit(approve, y_test, amount_test)\"\n")

---
Done? Go back to the lesson on [openfraudlabs.com/academy](https://openfraudlabs.com/academy/lesson/?course=data-science&n=33) and take the quiz.